In [ ]:
!git clone https://github.com/greatroboticslab/droneAI.git
%cd droneAI
!pip install yt-dlp pandas ultralytics
from google.colab import files
uploaded = files.upload()

import os
os.makedirs('videos', exist_ok=True)
os.makedirs('results', exist_ok=True)

student = "Mounika Varma"

real_links = ["https://www.youtube.com/watch?v=YZJKZjXpIB8"]
video_folder = './videos'
output_folder = './results'


from yt_dlp import YoutubeDL
import pandas as pd
import video_classifier as classifier
def download_video(video_link, video_folder):
    print("============================================")
    print("Downloading YouTube Video with cookies.txt...")
    print("============================================")
    video_path = f'{video_folder}/real_video1.mp4'
    ydl_opts = {
        'format': 'best',
        'outtmpl': video_path,
        'cookies': 'cookies.txt'
    }
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_link])
    return [video_path]

def detect_crashes(model, video_link, real_path):
    output_folder = './results'
    real_results = []

    for i, video_path in enumerate(real_path, start=1):
        crash_video_filename = f'Real_Flying_Video_{i}_Crashes.mp4'
        crash_video_path = os.path.join(output_folder, crash_video_filename)
        labeled_video_filename = f'Real_Flying_Video_{i}_Labeled.mp4'
        labeled_video_path = os.path.join(output_folder, labeled_video_filename)

        label_dict = {"Crash": 0, "Flight": 0, "No drone": 0, "No signal": 0, "No started": 0, "Started": 0, "Unstable": 0, "Landing": 0, "Unknown": 0}
        label_counts, unique_crashes, duration, total_frames = classifier.video_classification(label_dict, video_path, labeled_video_path, crash_video_path, model, 3.0)

        from main import VideoResult
        real_results.append(
            VideoResult(
                video_path=video_path,
                label_counts=label_counts,
                unique_crashes=unique_crashes,
                video_link=video_link,
                is_simulation=0,
                duration=duration,
                total_frames=total_frames,
                video_title=f"Real Flying #1"
            )
        )

    return real_results

def result_dataframe(student, real_results):
    data = []
    for result in real_results:
        total_labels = sum(result.label_counts.values())
        row = {
            "Student": student,
            "Video Instance": result.video_title,
            "Video Path": result.video_path,
            "Predicted Crashes": result.unique_crashes,
            "Video Link": result.video_link,
            "Is Simulation": result.is_simulation,
            "Video Duration": result.duration,
            "Total Frames": result.total_frames,
            "Crash Frames %": (result.label_counts["Crash"] / result.total_frames * 100),
            "Flight Frames %": (result.label_counts["Flight"] / total_labels * 100),
            "NoSignal Frames %": (result.label_counts["No signal"] / total_labels * 100),
            "Started Frames %": (result.label_counts["Started"] / total_labels * 100),
            "Landing Frames %": (result.label_counts["Landing"] / total_labels * 100)
        }
        data.append(row)
    return pd.DataFrame(data)

def run_model(video_link, real_paths, student, output_folder):
    print("============================================")
    print("Processing the video...")
    print("============================================")

    model = classifier.get_model()
    real_results = detect_crashes(model, video_link, real_paths)

    print("============================================")
    print("Saving results...")
    print("============================================")

    df_results = result_dataframe(student, real_results)
    df_results.to_csv(os.path.join(output_folder, 'results.csv'), index=False)
    print(" Results saved at: ./results/results.csv")

real_paths = download_video(real_links[0], video_folder)
run_model(real_links[0], real_paths, student, output_folder)


Cloning into 'droneAI'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 197 (delta 2), reused 23 (delta 2), pack-reused 172 (from 1)
Receiving objects: 100% (197/197), 70.43 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/droneAI
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 549.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.download('./results/Real_Flying_Video_1_Labeled.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('./results/results.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>